# Deployment Overview (Web Application)

This notebook focuses on deploying the **EduRecSys Hybrid Recommendation System**
as a **web-based application** powered by a RESTful API.

The deployment stage emphasizes **real-time recommendation inference**
and seamless integration with a web frontend, rather than offline evaluation
or experimental analysis.

---

### 🎯 Deployment Objective
The goal is to deliver a lightweight web application where users can:
- Enter or select a user identifier
- Receive personalized course recommendations instantly
- Interact with the system in a realistic, product-oriented manner

---

### 🌐 Deployment Platform
- **Application Type:** Web Application
- **Backend:** FastAPI (Python)
- **Frontend:** HTML / CSS / JavaScript
- **Interaction Model:** REST API

---

### 📌 Scope of Deployment

**Included Components:**
- Hybrid recommendation logic (Content-Based + Collaborative)
- Content representations using TF-IDF
- User profile construction from historical interactions
- Real-time recommendation generation via API

---

### 👤 Target Users

- **Existing Users:**  
  Users with historical interaction data stored in the system.

- **New Users (Cold Start):**  
  Users without prior interactions, handled via:
  - Popular course recommendations
  - Subject-based filtering
  - Language preference selection

---

### 🏗️ System Architecture

Web Browser  
↓  
Web Frontend (HTML / JavaScript)  
↓  
Recommendation API (FastAPI)  
↓  
Hybrid Recommendation Engine  
↓  
Ranked Course Recommendations

---

### 🔌 API Objective
The API is designed to:
- Accept a user identifier and optional parameters (e.g., number of recommendations)
- Return a ranked list of personalized course recommendations in JSON format
- Serve as the backend for a web-based user interface


In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import joblib


In [3]:
content_en = pd.read_csv("/kaggle/input/edurecsys-processed-content/content_en.csv")

content_en.head()

,content_id,title,description,subject,level,content_duration,language
0,1070968,Ultimate Investment Banking Course,Ultimate Investment Banking Course,Business Finance,All Levels,1.5,en
1,1113822,Complete GST Course & Certification - Grow You...,Complete GST Course & Certification - Grow You...,Business Finance,All Levels,39.0,en
2,1006314,Financial Modeling for Business Analysts and C...,Financial Modeling for Business Analysts and C...,Business Finance,Intermediate Level,2.5,en
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,Beginner to Pro - Financial Analysis in Excel ...,Business Finance,All Levels,3.0,en
4,1011058,How To Maximize Your Profits Trading Options,How To Maximize Your Profits Trading Options,Business Finance,Intermediate Level,2.0,en


In [4]:
content_columns = [
    "content_id",
    "title",
    "subject",
    "level"
]

content_deploy = content_en[content_columns].copy()
content_deploy.reset_index(drop=True, inplace=True)


In [5]:
content_deploy.head()

,content_id,title,subject,level
0,1070968,Ultimate Investment Banking Course,Business Finance,All Levels
1,1113822,Complete GST Course & Certification - Grow You...,Business Finance,All Levels
2,1006314,Financial Modeling for Business Analysts and C...,Business Finance,Intermediate Level
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,Business Finance,All Levels
4,1011058,How To Maximize Your Profits Trading Options,Business Finance,Intermediate Level


In [7]:
tfidf_vectorizer = joblib.load("/kaggle/input/edurecsys-models/tfidf_vectorizer.pkl")

content_similarity = joblib.load("/kaggle/input/edurecsys-models/content_similarity.pkl")

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
def recommend_cold_start(subject=None, level=None, k=5):
    df = content_deploy.copy()

    if subject is not None:
        df = df[df["subject"].str.lower() == subject.lower()]

    if level is not None:
        level_lower = level.lower()

        df["level_match"] = df["level"].str.lower().apply(
            lambda x: 1 if level_lower in x else 0
        )

        df = df.sort_values("level_match", ascending=False)

    if len(df) == 0:
        df = content_deploy.copy()

    return df.head(k)[["content_id", "title", "subject", "level"]].to_dict("records")


In [17]:
recommend_cold_start(
    subject="Business Finance",
    level="Beginner",
    k=5
)


[{'content_id': 364034,
  'title': 'Finanzas y Análisis Financiero: Manejo Seguro de Negocios',
  'subject': 'Business Finance',
  'level': 'Beginner Level'},
 {'content_id': 155344,
  'title': 'Introduction to Bookkeeping (Accounting)',
  'subject': 'Business Finance',
  'level': 'Beginner Level'},
 {'content_id': 941310,
  'title': 'Matemática Financeira com HP12C e MS Excel',
  'subject': 'Business Finance',
  'level': 'Beginner Level'},
 {'content_id': 1214144,
  'title': '¡Triunfar en La Bolsa de Valores No Requiere de Experiencia!',
  'subject': 'Business Finance',
  'level': 'Beginner Level'},
 {'content_id': 317572,
  'title': 'Accounting 1 Simplified for You',
  'subject': 'Business Finance',
  'level': 'Beginner Level'}]

In [22]:
def recommend_courses(
    user_id=None,
    subject=None,
    level=None,
    k=5
):
    """
    Unified recommendation entry point (Deployment version)
    """

    # Cold start 
    return {
        "mode": "cold_start",
        "recommendations": recommend_cold_start(
            subject=subject,
            level=level,
            k=k
        )
    }


In [23]:
recommend_courses(
    subject="Business Finance",
    level="Beginner",
    k=5
)


{'mode': 'cold_start',
 'recommendations': [{'content_id': 364034,
   'title': 'Finanzas y Análisis Financiero: Manejo Seguro de Negocios',
   'subject': 'Business Finance',
   'level': 'Beginner Level'},
  {'content_id': 155344,
   'title': 'Introduction to Bookkeeping (Accounting)',
   'subject': 'Business Finance',
   'level': 'Beginner Level'},
  {'content_id': 941310,
   'title': 'Matemática Financeira com HP12C e MS Excel',
   'subject': 'Business Finance',
   'level': 'Beginner Level'},
  {'content_id': 1214144,
   'title': '¡Triunfar en La Bolsa de Valores No Requiere de Experiencia!',
   'subject': 'Business Finance',
   'level': 'Beginner Level'},
  {'content_id': 317572,
   'title': 'Accounting 1 Simplified for You',
   'subject': 'Business Finance',
   'level': 'Beginner Level'}]}